# 01 — EDA do Pipeline de Vendas

**Dataset:** 5 CSVs do CRM Sales Predictive Analytics (Kaggle, CC0)  
**Objetivo:** Derivar os thresholds e pesos do score de priorização a partir dos dados reais — não de suposições.

> Cada finding numerado corresponde diretamente a um componente do scoring engine em `src/lib/queries.ts`.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)

# ---------- Load ----------
pipeline  = pd.read_csv('../data/sales_pipeline.csv')
products  = pd.read_csv('../data/products.csv')
accounts  = pd.read_csv('../data/accounts.csv')
teams     = pd.read_csv('../data/sales_teams.csv')

print(f'Pipeline rows  : {len(pipeline):,}')
print(f'Products rows  : {len(products):,}')
print(f'Accounts rows  : {len(accounts):,}')
print(f'Teams rows     : {len(teams):,}')
pipeline.head(3)

In [ ]:
# Normalizar nome de produto (GTXPro → GTX Pro)
pipeline['product_norm'] = pipeline['product'].str.replace('GTXPro', 'GTX Pro', regex=False)

# Merge principal
df = (
    pipeline
    .merge(products.rename(columns={'product': 'product_norm'}), on='product_norm', how='left')
    .merge(accounts, on='account', how='left')
    .merge(teams, on='sales_agent', how='left')
)

# Deals abertos (pipeline ativo)
open_df = df[df['deal_stage'].isin(['Prospecting', 'Engaging'])].copy()

# Dataset de referência para win rate (histórico Won/Lost)
closed_df = df[df['deal_stage'].isin(['Won', 'Lost'])].copy()

# Aging: dataset histórico termina em 2017-12-31
REF_DATE = pd.Timestamp('2017-12-31')
open_df['engage_date'] = pd.to_datetime(open_df['engage_date'], errors='coerce')
open_df['days_in_pipeline'] = (REF_DATE - open_df['engage_date']).dt.days

print(f'\nDeals abertos (Prospecting + Engaging): {len(open_df):,}')
print(f'Deals históricos (Won + Lost)          : {len(closed_df):,}')

## Finding 1 — Composição do Pipeline

**Pergunta:** Quantos deals estão ativos e como se distribuem entre stages?

In [ ]:
stage_counts = open_df['deal_stage'].value_counts()
total_open = len(open_df)

print('=== FINDING 1 — COMPOSIÇÃO DO PIPELINE ===')
print(f'Total de deals abertos: {total_open:,}')
for stage, count in stage_counts.items():
    print(f'  {stage:15s}: {count:,} ({count/total_open*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#2563eb', '#f59e0b']
stage_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Deals Abertos por Stage', fontweight='bold')
ax.set_xlabel('Stage')
ax.set_ylabel('Número de Deals')
ax.tick_params(axis='x', rotation=0)
for bar in ax.patches:
    ax.annotate(f'{int(bar.get_height()):,}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n→ Decisão de scoring: Engaging = 30 pts (maior engajamento), Prospecting = 15 pts')

## Finding 2 — Distribuição de Preço dos Produtos

**Pergunta:** Quais são os tiers naturais de preço que justificam os thresholds de scoring?

In [ ]:
print('=== FINDING 2 — DISTRIBUIÇÃO DE PREÇO ===')
price_by_product = (
    products.rename(columns={'product': 'product_norm'})
    .assign(product_norm=lambda x: x['product_norm'].str.replace('GTXPro', 'GTX Pro', regex=False))
    [['product_norm', 'series', 'sales_price']]
    .drop_duplicates()
    .sort_values('sales_price', ascending=False)
)
print(price_by_product.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart de preço por produto
bar_colors = {'GTK': '#10b981', 'GTX': '#2563eb', 'MG': '#94a3b8'}
colors_list = [bar_colors.get(s, '#999') for s in price_by_product['series']]
axes[0].bar(price_by_product['product_norm'], price_by_product['sales_price'], color=colors_list, edgecolor='white')
axes[0].set_title('Preço por Produto', fontweight='bold')
axes[0].set_ylabel('Sales Price ($)')
axes[0].tick_params(axis='x', rotation=40)
# Threshold lines
for thresh, label in [(4000, '$4k'), (1000, '$1k'), (500, '$500')]:
    axes[0].axhline(thresh, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
    axes[0].text(len(price_by_product)-0.5, thresh+30, label, color='red', fontsize=9)

# Histograma de preço nos deals abertos
open_prices = open_df['sales_price'].dropna()
axes[1].hist(open_prices, bins=20, color='#2563eb', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribuição de Preço — Deals Abertos', fontweight='bold')
axes[1].set_xlabel('Sales Price ($)')
axes[1].set_ylabel('Frequência')
for thresh in [500, 1000, 4000]:
    axes[1].axvline(thresh, color='red', linestyle='--', linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.show()

# Contagem por tier
tiers = [
    ('≥ $4.000 (premium)', open_prices >= 4000),
    ('$1.000–$3.999', (open_prices >= 1000) & (open_prices < 4000)),
    ('$500–$999', (open_prices >= 500) & (open_prices < 1000)),
    ('< $500 (entrada)', open_prices < 500),
]
print('\nDeals por tier de preço:')
for label, mask in tiers:
    n = mask.sum()
    print(f'  {label:25s}: {n:,} ({n/len(open_prices)*100:.1f}%)')

print('\n→ GTX Pro é o único produto acima de $4k (outlier premium)')
print('→ Thresholds $4k/$1k/$500 separam tiers naturais do catálogo')

## Finding 3 — Win Rate dos Agentes

**Pergunta:** Qual a distribuição real de win rate? Onde está o baseline justo para normalização?

In [ ]:
agent_stats = (
    closed_df
    .groupby('sales_agent')
    .apply(lambda x: pd.Series({
        'won':      (x['deal_stage'] == 'Won').sum(),
        'lost':     (x['deal_stage'] == 'Lost').sum(),
        'total':    len(x),
        'win_rate': (x['deal_stage'] == 'Won').mean() * 100
    }), include_groups=False)
    .reset_index()
    .sort_values('win_rate', ascending=False)
)

print('=== FINDING 3 — WIN RATE DOS AGENTES ===')
print(f'Agentes com histórico: {len(agent_stats)}')
print(f'Win rate — min: {agent_stats["win_rate"].min():.1f}%  |  max: {agent_stats["win_rate"].max():.1f}%  |  média: {agent_stats["win_rate"].mean():.1f}%')
print(f'Quartis: Q1={agent_stats["win_rate"].quantile(.25):.1f}%  Q2={agent_stats["win_rate"].quantile(.5):.1f}%  Q3={agent_stats["win_rate"].quantile(.75):.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histograma
axes[0].hist(agent_stats['win_rate'], bins=15, color='#10b981', edgecolor='white', alpha=0.8)
axes[0].axvline(55, color='red', linestyle='--', label='Baseline (55%)')
axes[0].axvline(agent_stats['win_rate'].mean(), color='orange', linestyle='-', label=f'Média ({agent_stats["win_rate"].mean():.1f}%)')
axes[0].axvline(70, color='purple', linestyle='--', label='Teto (70%)')
axes[0].set_title('Distribuição de Win Rate — Agentes', fontweight='bold')
axes[0].set_xlabel('Win Rate (%)')
axes[0].set_ylabel('Agentes')
axes[0].legend(fontsize=9)

# Bar chart top/bottom
top10 = agent_stats.head(10)
axes[1].barh(top10['sales_agent'], top10['win_rate'], color='#2563eb', edgecolor='white')
axes[1].axvline(55, color='red', linestyle='--', linewidth=0.8)
axes[1].set_title('Top 10 Agentes por Win Rate', fontweight='bold')
axes[1].set_xlabel('Win Rate (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('\n→ Faixa 55–70%: cobre praticamente toda a distribuição real')
print('→ Baseline 55% = piso sem penalizar agentes medianos')
print('→ Normalização linear (win_rate - 55) / 15 * 10 → score 0–10')

## Finding 4 — Quartis de Receita das Contas

**Pergunta:** Quais são os quartis da receita das contas presentes no pipeline?

In [ ]:
# Revenue em M$ (já está em milhões no CSV)
rev = accounts['revenue'].dropna().astype(float)

q1  = rev.quantile(0.25)
q2  = rev.quantile(0.50)
q3  = rev.quantile(0.75)

print('=== FINDING 4 — QUARTIS DE RECEITA DE CONTA ===')
print(f'Q1 (25th pct): ${q1:,.0f}M')
print(f'Q2 (50th pct): ${q2:,.0f}M  (mediana)')
print(f'Q3 (75th pct): ${q3:,.0f}M')
print(f'Min: ${rev.min():,.0f}M  |  Max: ${rev.max():,.0f}M')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histograma
axes[0].hist(rev, bins=30, color='#f59e0b', edgecolor='white', alpha=0.8)
for q, label, color in [(q1, f'Q1 ${q1:,.0f}M', '#ef4444'), (q2, f'Q2 ${q2:,.0f}M', '#f97316'), (q3, f'Q3 ${q3:,.0f}M', '#8b5cf6')]:
    axes[0].axvline(q, color=color, linestyle='--', linewidth=1.5, label=label)
axes[0].set_title('Distribuição de Receita das Contas ($M)', fontweight='bold')
axes[0].set_xlabel('Revenue ($M)')
axes[0].set_ylabel('Contas')
axes[0].legend(fontsize=9)

# Deals por tier de receita (no pipeline aberto)
open_rev = open_df['revenue'].dropna().astype(float)
tier_labels = [f'≤ Q1\n(≤${q1:,.0f}M)', f'Q1–Q2\n(${q1:,.0f}–${q2:,.0f}M)', f'Q2–Q3\n(${q2:,.0f}–${q3:,.0f}M)', f'> Q3\n(>${q3:,.0f}M)']
tier_counts = [
    (open_rev <= q1).sum(),
    ((open_rev > q1) & (open_rev <= q2)).sum(),
    ((open_rev > q2) & (open_rev <= q3)).sum(),
    (open_rev > q3).sum(),
]
axes[1].bar(tier_labels, tier_counts, color=['#94a3b8', '#60a5fa', '#f59e0b', '#10b981'], edgecolor='white')
axes[1].set_title('Deals Abertos por Tier de Receita da Conta', fontweight='bold')
axes[1].set_ylabel('Deals')
for i, v in enumerate(tier_counts):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n→ Q1=${q1:,.0f}M, Q2={q2:,.0f}M, Q3={q3:,.0f}M são os thresholds exatos do componente score_account')
print('→ Pontuação: >Q3=20pts, >Q2=14pts, >Q1=8pts, ≤Q1=4pts')

## Finding 5 — Distribuição de Aging (Dias no Pipeline)

**Pergunta:** Onde estão os breaks naturais da distribuição de tempo no pipeline?

In [ ]:
aging = open_df['days_in_pipeline'].dropna()

print('=== FINDING 5 — AGING DISTRIBUTION ===')
print(f'Média  : {aging.mean():.0f} dias')
print(f'Mediana: {aging.median():.0f} dias')
print(f'P75    : {aging.quantile(0.75):.0f} dias')
print(f'P90    : {aging.quantile(0.90):.0f} dias')
print(f'Max    : {aging.max():.0f} dias')

print('\nDeals por faixa de aging:')
buckets = [(0, 120, '0–120 dias'), (121, 200, '121–200 dias'), (201, 300, '201–300 dias'), (301, 9999, '> 300 dias')]
for low, high, label in buckets:
    mask = (aging >= low) & (aging <= high)
    n = mask.sum()
    print(f'  {label:15s}: {n:,} ({n/len(aging)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histograma com linhas de threshold
axes[0].hist(aging, bins=40, color='#6366f1', edgecolor='white', alpha=0.8)
for thresh, label in [(120, '120d −5pts'), (200, '200d −10pts'), (300, '300d −15pts')]:
    axes[0].axvline(thresh, color='red', linestyle='--', linewidth=1.2, label=label)
axes[0].set_title('Distribuição de Aging — Deals Abertos', fontweight='bold')
axes[0].set_xlabel('Dias no Pipeline')
axes[0].set_ylabel('Deals')
axes[0].legend(fontsize=9)

# Boxplot
axes[1].boxplot(aging, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#6366f1', alpha=0.6),
                medianprops=dict(color='white', linewidth=2))
for thresh in [120, 200, 300]:
    axes[1].axhline(thresh, color='red', linestyle='--', linewidth=0.8)
axes[1].set_title('Boxplot de Aging', fontweight='bold')
axes[1].set_ylabel('Dias no Pipeline')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

print('\n→ Breaks em 120/200/300 dias correspondem a inflexões naturais na distribuição')
print('→ Penalidades: >120d=-5, >200d=-10, >300d=-15 (deterioração progressiva)')

## Finding 6 — Composição por Série e Win Rate Histórico

**Pergunta:** Quais séries têm maior win rate histórico?

In [ ]:
closed_with_series = closed_df.copy()
closed_with_series['product_norm'] = closed_with_series['product'].str.replace('GTXPro', 'GTX Pro', regex=False)
closed_with_series = closed_with_series.merge(
    products.rename(columns={'product': 'product_norm'})[['product_norm', 'series']],
    on='product_norm', how='left'
)

series_stats = (
    closed_with_series
    .groupby('series')
    .apply(lambda x: pd.Series({
        'total_deals':  len(x),
        'won':          (x['deal_stage'] == 'Won').sum(),
        'lost':         (x['deal_stage'] == 'Lost').sum(),
        'win_rate_pct': (x['deal_stage'] == 'Won').mean() * 100,
    }), include_groups=False)
    .reset_index()
    .sort_values('win_rate_pct', ascending=False)
)

print('=== FINDING 6 — WIN RATE POR SÉRIE ===')
print(series_stats[['series', 'total_deals', 'won', 'lost', 'win_rate_pct']].round(1).to_string(index=False))

# Composição no pipeline aberto
open_with_series = open_df.copy()
open_series_counts = open_df['series'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bar_colors = {'GTK': '#10b981', 'GTX': '#2563eb', 'MG': '#94a3b8'}

# Win rate por série
colors = [bar_colors.get(s, '#999') for s in series_stats['series']]
axes[0].bar(series_stats['series'], series_stats['win_rate_pct'], color=colors, edgecolor='white')
axes[0].set_title('Win Rate Histórico por Série (%)', fontweight='bold')
axes[0].set_ylabel('Win Rate (%)')
for i, (_, row) in enumerate(series_stats.iterrows()):
    axes[0].text(i, row['win_rate_pct'] + 0.3, f"{row['win_rate_pct']:.1f}%", ha='center', fontsize=11, fontweight='bold')

# Composição no pipeline aberto
open_colors = [bar_colors.get(s, '#999') for s in open_series_counts.index]
axes[1].bar(open_series_counts.index, open_series_counts.values, color=open_colors, edgecolor='white')
axes[1].set_title('Deals Abertos por Série', fontweight='bold')
axes[1].set_ylabel('Deals')
for i, (series, count) in enumerate(open_series_counts.items()):
    axes[1].text(i, count + 5, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n→ GTK lidera em win rate → 15 pts')
print('→ GTX segundo lugar → 8 pts')
print('→ MG menor win rate → 0 pts (não penaliza, apenas não acrescenta)')

## Finding 7 — Distribuição Final do Score

Resultado: como o score se distribui nos 2.089 deals abertos após aplicar os 6 componentes.

In [ ]:
# Replicar o scoring do SQL em Python para validação

# Win rate por agente (do histórico)
wr = (
    closed_df
    .groupby('sales_agent')
    .apply(lambda x: round((x['deal_stage'] == 'Won').mean() * 100, 1), include_groups=False)
    .rename('win_rate_pct')
    .reset_index()
)

scored = open_df.merge(wr, on='sales_agent', how='left')
scored['win_rate_pct'] = scored['win_rate_pct'].fillna(62.5)
scored['revenue'] = scored['revenue'].fillna(0).astype(float)
scored['sales_price'] = scored['sales_price'].fillna(0).astype(float)

# 6 componentes
scored['score_stage']   = scored['deal_stage'].map({'Engaging': 30, 'Prospecting': 15}).fillna(0)
scored['score_value']   = pd.cut(scored['sales_price'], bins=[-1, 499, 999, 3999, 99999], labels=[3, 8, 15, 25]).astype(float)
scored['score_account'] = pd.cut(scored['revenue'], bins=[-1, 497, 1224, 2741, 999999], labels=[4, 8, 14, 20]).astype(float)
scored['score_time']    = pd.cut(scored['days_in_pipeline'], bins=[-1, 120, 200, 300, 99999], labels=[0, -5, -10, -15]).astype(float)
scored['score_series']  = scored['series'].map({'GTK': 15, 'GTX': 8, 'MG': 0}).fillna(0)
scored['score_agent']   = scored['win_rate_pct'].apply(lambda r: round(min(10, max(0, (r - 55) / 15 * 10)), 1))

scored['score'] = (
    scored[['score_stage', 'score_value', 'score_account', 'score_time', 'score_series', 'score_agent']].sum(axis=1)
).clip(0, 100)

hot  = (scored['score'] >= 70).sum()
warm = ((scored['score'] >= 40) & (scored['score'] < 70)).sum()
cold = (scored['score'] < 40).sum()

print('=== FINDING 7 — DISTRIBUIÇÃO DO SCORE FINAL ===')
print(f'Total de deals avaliados: {len(scored):,}')
print(f'Hot  (≥70): {hot:,}  ({hot/len(scored)*100:.1f}%)')
print(f'Warm (40–69): {warm:,}  ({warm/len(scored)*100:.1f}%)')
print(f'Cold (<40): {cold:,}  ({cold/len(scored)*100:.1f}%)')
print(f'Média: {scored["score"].mean():.1f}  |  Mediana: {scored["score"].median():.1f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histograma do score
axes[0].hist(scored['score'], bins=30, color='#6366f1', edgecolor='white', alpha=0.8)
axes[0].axvline(40, color='#f59e0b', linestyle='--', linewidth=1.5, label='Warm threshold (40)')
axes[0].axvline(70, color='#ef4444', linestyle='--', linewidth=1.5, label='Hot threshold (70)')
axes[0].set_title('Distribuição do Score Final (0–100)', fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Deals')
axes[0].legend(fontsize=9)

# Pizza Hot/Warm/Cold
axes[1].pie([hot, warm, cold],
            labels=[f'Hot\n{hot} deals', f'Warm\n{warm} deals', f'Cold\n{cold} deals'],
            colors=['#ef4444', '#f59e0b', '#6366f1'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Pipeline por Classificação', fontweight='bold')

plt.tight_layout()
plt.show()

## Conclusão — Thresholds Validados

Sumário de como cada finding derivou um componente do score.

In [ ]:
print('=== THRESHOLDS DERIVADOS DOS DADOS ===')
print()
print('Finding 1 → score_stage')
print('  Engaging = 30 pts, Prospecting = 15 pts')
print('  Razão: Engaging indica deal mais avançado no funil')
print()
print('Finding 2 → score_value')
print('  Thresholds $4k/$1k/$500 separam tiers naturais do catálogo')
print('  GTX Pro ($4.821) é o único produto premium acima de $4k')
print()
print('Finding 3 → score_agent')
print(f'  Win rate real: 55–70%, média {agent_stats["win_rate"].mean():.1f}%')
print('  Normalização linear (r−55)/15×10 → 0–10 pts')
print()
print('Finding 4 → score_account')
print(f'  Q1=${q1:,.0f}M → 8 pts, Q2=${q2:,.0f}M → 14 pts, Q3={q3:,.0f}M → 20 pts')
print('  Quartis reais de accounts.csv')
print()
print('Finding 5 → score_time (penalidades)')
print('  Breaks em 120/200/300 dias: inflexões naturais da distribuição')
print('  Penalidades: −5/−10/−15 pts progressivos')
print()
print('Finding 6 → score_series')
print(f'  GTK win rate: {series_stats[series_stats["series"]=="GTK"]["win_rate_pct"].values[0]:.1f}% → 15 pts')
print(f'  GTX win rate: {series_stats[series_stats["series"]=="GTX"]["win_rate_pct"].values[0]:.1f}% → 8 pts')
print(f'  MG  win rate: {series_stats[series_stats["series"]=="MG"]["win_rate_pct"].values[0]:.1f}% → 0 pts')